# 02 — Model Build

This notebook is what actually produces `models/flight_delay_model.pkl`: it cross-validates the three model families on an expanding window, fits the winning family on the full development set, and exports the artefact `streamlit_app.py` reads. Everything here calls into `app.*` — no logic is reimplemented in this notebook.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from app import config
from app.models import train
from app.models.zoo import FOREST_CONFIG, model_zoo

frame = pd.read_parquet(config.FEATURES_PARQUET)
print(f"feature rows: {len(frame):,}")
print(f"periods: {sorted(frame.period.unique())}")

feature rows: 5,696,742
periods: ['2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']


## The three model families

`linear` is the interpretable floor. `forest` runs deliberately shrunk (100 trees, depth 10) rather than a tuned configuration — it is the slowest family and does not beat boosting on any metric, so the reduced settings are reported here rather than hidden. `boosted` is LightGBM, which handles the categorical explosion natively.

In [3]:
print("forest config (deliberately shrunk):", FOREST_CONFIG)
for name, model in model_zoo().items():
    print(f"{name:8} -> {model}")

forest config (deliberately shrunk): {'n_estimators': 100, 'max_depth': 10}
linear   -> Pipeline(steps=[('scale', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000, random_state=42))])
forest   -> RandomForestClassifier(max_depth=10, n_jobs=-1, random_state=42)
boosted  -> LGBMClassifier(learning_rate=0.05, n_estimators=400, n_jobs=-1, num_leaves=63,
               random_state=42, verbose=-1)


## Cross-validation on expanding windows

Two of the five configured folds, run here for a fast, genuine demonstration of the expanding-window methodology (never random k-fold, since the history features are expanding means over prior months). The full 5-fold result is recorded in `models/train_log.txt` from the complete run.

In [4]:
results = train.run_cv(frame, validation_periods=["2026-03", "2026-04"])
summary = train.summarise(results)
summary

,pr_auc,pr_auc_sd,lift,recall,precision,roc_auc,brier,fit_s
model,,,,,,,,
boosted,0.3512,0.0338,1.5816,0.2971,0.3944,0.6676,0.1646,296.95
forest,0.3451,0.0362,1.5532,0.1982,0.4130,0.6634,0.1652,209.70
linear,0.3232,0.0335,1.4546,0.1981,0.3768,0.6464,0.1682,16.80


## Fit the winning family on the full development set

Whichever family led the cross-validation above is fit one more time — on every development-period row, not just the folds — and exported. This is the step that (re)writes `models/flight_delay_model.pkl`.

In [5]:
best_family = summary.index[0]
print(f"best family by mean lift: {best_family}")

model, meta = train.fit_final(frame, best_family)
print(f"fit on {meta['rows']:,} rows, {meta['period_start']} to {meta['period_end']}, {meta['fit_s']}s")

best family by mean lift: boosted
fit on 5,696,742 rows, 2025-07 to 2026-04, 114.2s


In [6]:
artefact_path = train.export(model, frame, results, family=best_family, target_recall=0.60)
print(f"artefact written to: {artefact_path}")

artefact written to: D:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\projects\capstone_1\models\flight_delay_model.pkl


## Verify the exported artefact loads and scores

The same check `streamlit_app.py` does on startup: load the pickle, score one flight.

In [7]:
from app.serving.lookups import reset_cache
from app.serving.predict import predict_flight

reset_cache()  # this session may have cached lookups built from a stale run
result = predict_flight(
    carrier="DL", origin="ATL", dest="ORD", departure_hour=18,
    day_of_week=5, month=7, distance_miles=606, scheduled_minutes=115, leg_number=3,
)
print(result.summary())
print(f"risk band: {result.risk_band}")
for reason in result.top_reasons:
    print(f" - {reason}")

34% chance of arriving 15+ min late - FLAG for attention
risk band: high
 - how much history exists for this route raises the risk
 - Friday raises the risk
 - the scheduled departure time raises the risk
 - how often the arrival airport runs late is 28% - this raises the risk
